# Goodness-of-Fit for a Likelihood Fit using Simulation

## Description:

A $\chi^2$ fit comes with a goodness-of-fit (GoF) measure **for free**: the value of
$\chi^2$ at the minimum follows (approximately) a $\chi^2$ distribution with
$N_\mathrm{dof}$ degrees of freedom. Looking this value up immediately gives a
**p-value**, telling us how probable it is to get a fit *this bad or worse* if the model
is correct.

An **(unbinned) Maximum Likelihood fit gives no such number**. The value of $-2\ln L$ at
the minimum is *not* in general $\chi^2$-distributed (it depends very much on the PDF shape,
and can easily come out negative!), so it cannot simply be looked up in
a table to get a p-value. This is a very common point of confusion: a likelihood fit
happily returns the best parameters, but says nothing about whether the model actually
describes the data.

This exercise shows how to obtain a goodness-of-fit p-value for a likelihood fit using
**simulation ("toy" Monte Carlo)** by the following recipe:

1. Fit the data and read off the best-fit parameters.
2. Simulate many (here 1000) **toy datasets drawn from the fitted PDF**.
3. Re-fit each toy and record the fit statistic ($-2\ln L$ at the minimum).
4. The position (i.e. quantile) of the **observed** value inside this simulated distribution is the p-value. 

The price is computational: instead of one fit we do a thousand. The reward is a valid
GoF p-value, and as a bonus a direct, honest picture of how well our estimator behaves.

We use a 1D dataset of $\sim$1000 events from a **5-parameter** model: a Gaussian signal
peak sitting in the **low-statistics tail** of a falling (exponential + flat) background.
That tail is exactly where a binned $\chi^2$ fit runs into trouble (empty / low-count
bins) and where the unbinned likelihood gives the better parameter estimate.

### References:
- Barlow, chapter 8 (goodness of fit)
- Cowan, chapter 7

### Author:
- Troels C. Petersen (Niels Bohr Institute)

### Date:
- 27-10-2026 (latest version)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from iminuit import Minuit, cost
from scipy import stats

r = np.random            # Shorthand for the random generator
r.seed(42)               # Fixed seed: random, but reproducible from run to run
plt.rcParams['font.size'] = 16

## 1. The problem and the dataset

We generate $N = 1000$ values on the range $[x_\mathrm{min}, x_\mathrm{max}] = [0, 20]$
from a **mixture of three components** (a proper PDF, normalised to 1 over the range):

$$ f(x) = N \left[ f_g\,G(x;\mu,\sigma) \;+\; f_f\, / (x_\mathrm{max}-x_\mathrm{min})
        \;+\; (1-f_g-f_f)\, e^{-x/\tau} / \tau \right] $$

i.e. a **Gaussian signal peak** ($f_g, \mu, \sigma$) on top of a **flat** ($f_f$) plus
**exponential** ($\tau$) background — **five free parameters**. Each component is
individually normalised over the fit range.

The peak is small (only $f_g \approx 4\%$, about 40 events) and sits at $\mu = 14$, out in
the **low-statistics tail** where the steep background has fallen to a handful of events
per bin (and several bins are empty). Estimating such a faint peak is exactly where
unbinned likelihood beats a binned $\chi^2$, and where the $\chi^2$ goodness-of-fit
approximation starts to fray.

In [ ]:
# --- Parameters of the problem (a 5-parameter model) ---
Nevents    = 1000       # Number of events in the dataset
xmin, xmax = 0.0, 20.0  # Fit range

# True parameters:  signal Gaussian + flat + exponential background
#   f_g : signal fraction,  f_f : flat fraction,  mu/sigma : peak,  tau : decay
# A small (~40 event) peak sitting on a sparse background out in the tail:
par_names = ['f_g', 'f_f', 'mu', 'sigma', 'tau']
par_true  = (0.04, 0.03, 14.0, 0.7, 3.0)

# Mixture PDF on [xmin, xmax], normalised to 1 (a proper density):
def model_pdf(x, f_g, f_f, mu, sigma, tau):
    lo, hi = stats.norm.cdf([xmin, xmax], mu, sigma)
    pdf_g  = stats.norm.pdf(x, mu, sigma) / max(hi - lo, 1e-12)
    pdf_f  = 1.0 / (xmax - xmin)
    norm_e = tau * (np.exp(-xmin/tau) - np.exp(-xmax/tau))
    pdf_e  = np.exp(-x/tau) / norm_e
    return f_g*pdf_g + f_f*pdf_f + (1.0 - f_g - f_f)*pdf_e

# Draw N values from the mixture (each component by its fraction):
def sample_model(par, N):
    f_g, f_f, mu, sigma, tau = par
    n_g, n_f, n_e = r.multinomial(N, [f_g, f_f, 1.0 - f_g - f_f])
    lo, hi = stats.norm.cdf([xmin, xmax], mu, sigma)
    xg = stats.norm.ppf(lo + r.uniform(size=n_g)*(hi - lo), mu, sigma)
    xf = r.uniform(xmin, xmax, size=n_f)
    a, b = np.exp(-xmin/tau), np.exp(-xmax/tau)
    xe = -tau * np.log(a - r.uniform(size=n_e)*(a - b))
    return np.concatenate([xg, xf, xe])

In [ ]:
data = sample_model(par_true, Nevents)

# Histogram settings used throughout:
Nbins    = 50
bin_edges = np.linspace(xmin, xmax, Nbins+1)
binwidth  = (xmax - xmin) / Nbins
bin_centers = 0.5*(bin_edges[:-1] + bin_edges[1:])
counts, _ = np.histogram(data, bins=bin_edges)

print(f"Total events            : {Nevents}")
print(f"Events in tail (x > 12) : {(data > 12).sum()}")
print(f"Empty bins              : {(counts == 0).sum()} out of {Nbins}")

fig, ax = plt.subplots(figsize=(12, 6))
ax.errorbar(bin_centers, counts, yerr=np.sqrt(counts), fmt='k.', label='Data')
ax.axvspan(12, xmax, alpha=0.15, color='red', label='Low-statistics tail')
ax.set(xlabel='x', ylabel=f'Counts / {binwidth:.1f}',
       title='Signal peak on a falling background (1000 events)')
ax.legend()
fig.tight_layout()

## 2. The unbinned Maximum Likelihood fit

We fit the data with an **unbinned** likelihood, using each individual measurement (no
binning, no loss of information — ideal for the sparse peak region). `iminuit`'s
`cost.UnbinnedNLL` builds the negative log-likelihood
$-2\ln L = -2\sum_i \ln f(x_i)$ directly from the data and our (normalised) PDF, and fits
all five parameters at once.

Note what we get back: excellent parameter estimates **and the value of $-2\ln L$ at the
minimum** — but, crucially, no p-value. The number `m.fval` on its own does not tell us
whether the fit is good or bad.

In [ ]:
nll = cost.UnbinnedNLL(data, model_pdf)
m_ull = Minuit(nll, f_g=0.05, f_f=0.03, mu=14.0, sigma=1.0, tau=3.0)
m_ull.limits['f_g'] = (0, 1); m_ull.limits['f_f'] = (0, 1)
m_ull.limits['sigma'] = (0.1, None); m_ull.limits['tau'] = (0.1, None)
m_ull.limits['mu'] = (xmin, xmax)
m_ull.migrad()

In [ ]:
par_ull = np.array(m_ull.values)
T_obs   = m_ull.fval            # -2 ln L at the minimum: our GoF test statistic
print("Unbinned ML fit:")
for n in par_names:
    print(f"  {n:5s} = {m_ull.values[n]:7.3f} +/- {m_ull.errors[n]:.3f}")
print(f"\n  -2ln(L) at minimum = {T_obs:.3f}")
print("  NOTE: This number does not tell you whether the fit is good or not!")

## 3. The $\chi^2$ fit — goodness-of-fit comes for free

For comparison we also do a **binned $\chi^2$ fit**. Here the model predicts the expected
number of counts in each bin, and we minimise

$$ \chi^2 = \sum_{\text{bins}} \frac{(n_i - \nu_i(\tau))^2}{\sigma_i^2}, \qquad \sigma_i = \sqrt{n_i}. $$

The $\sqrt{n_i}$ error is only defined for **non-empty** bins, so we must drop the empty
bins in the tail — already a warning sign. The pay-off is the built-in GoF: at the minimum
$\chi^2$ follows a $\chi^2$ distribution with $N_\mathrm{dof} = N_\mathrm{bins,used} -
N_\mathrm{par}$, and `scipy.stats.chi2.sf` turns it straight into a p-value.

In [ ]:
# Model for the EXPECTED COUNTS in a bin (extra parameter N = total normalisation):
def model_counts(x, N, f_g, f_f, mu, sigma, tau):
    return N * binwidth * model_pdf(x, f_g, f_f, mu, sigma, tau)

mask = counts > 0    # chi2 with sqrt(N) errors is only defined for non-empty bins
ls = cost.LeastSquares(bin_centers[mask], counts[mask], np.sqrt(counts[mask]), model_counts)
m_chi2 = Minuit(ls, N=Nevents, f_g=0.05, f_f=0.03, mu=14.0, sigma=1.0, tau=3.0)
m_chi2.limits['f_g'] = (0, 1); m_chi2.limits['f_f'] = (0, 1)
m_chi2.limits['sigma'] = (0.1, None); m_chi2.limits['tau'] = (0.1, None)
m_chi2.migrad()

chi2_val  = m_chi2.fval
Ndof      = int(mask.sum() - m_chi2.nfit)
chi2_prob = stats.chi2.sf(chi2_val, Ndof)
print(f"Chi2 fit :  f_g = {m_chi2.values['f_g']:.3f} +/- {m_chi2.errors['f_g']:.3f}")
print(f"  P(chi2={chi2_val:.1f}, Ndof={Ndof:2d}) = {chi2_prob:.3f}")

In [ ]:
xcurve = np.linspace(xmin, xmax, 500)
fig, ax = plt.subplots(figsize=(12, 6))
ax.errorbar(bin_centers, counts, yerr=np.sqrt(counts), fmt='k.', label='Data')
ax.plot(xcurve, Nevents*binwidth*model_pdf(xcurve, *par_ull), '-', label='Unbinned LL')
ax.plot(xcurve, model_counts(xcurve, *m_chi2.values), '--', label='Chi2 binned')
ax.set(xlabel='x', ylabel=f'Counts / {binwidth:.1f}', yscale='log',
       title='Both fits on a log scale (note the peak in the sparse tail)')
ax.legend()
fig.tight_layout()

## 4. Goodness-of-fit for the likelihood fit, by simulation

Now, we want to know **if $T_\mathrm{obs} = -2\ln L_\mathrm{min}$ is a
typical value if our model is correct, or is it surprisingly large (i.e. a bad fit)?**

Since we cannot look the answer up in a table, we *build the table/distributions ourselves* by
simulation:

1. Treat the fitted PDF as the truth.
2. Draw a toy dataset of the **same size** $N = 1000$ from it.
3. Re-fit the toy and store its $-2\ln L_\mathrm{min}$.
4. Repeat 1000 times → the **sampling distribution** of the test statistic *under the
   hypothesis that the model is correct*.

The p-value is then simply the fraction of toys with a statistic at least as large as the
observed one. Keeping $N$ fixed across toys makes the $-2\ln L$ values directly comparable.

In the same loop we also re-fit each toy with the $\chi^2$ method, so we can later compare
the two estimators — simulation gives us all of this at once.

In [ ]:
N_toy = 1000     # Number of simulated datasets (1000 -> a smooth, well-sampled distribution)

toy_2lnL    = np.zeros(N_toy)   # GoF statistic (-2 ln L at minimum), unbinned fit
toy_fg_ull  = np.zeros(N_toy)   # fitted signal fraction, unbinned likelihood
toy_fg_chi2 = np.zeros(N_toy)   # fitted signal fraction, binned chi2
toy_chi2    = np.zeros(N_toy)   # chi2 value of each toy fit
toy_ndof    = np.zeros(N_toy)   # ndof of each toy chi2 fit
p0 = dict(zip(par_names, par_ull))   # start each toy fit at the fitted parameters

for i in range(N_toy):
    toy = sample_model(par_ull, Nevents)      # simulate FROM THE FITTED PDF

    # --- unbinned ML fit of the toy ---
    m = Minuit(cost.UnbinnedNLL(toy, model_pdf), **p0)
    m.limits['f_g'] = (0, 1); m.limits['f_f'] = (0, 1)
    m.limits['sigma'] = (0.1, None); m.limits['tau'] = (0.1, None); m.limits['mu'] = (xmin, xmax)
    m.migrad()
    toy_2lnL[i]   = m.fval
    toy_fg_ull[i] = m.values['f_g']

    # --- binned chi2 fit of the SAME toy (for comparison) ---
    c, _ = np.histogram(toy, bins=bin_edges)
    msk  = c > 0
    mc = Minuit(cost.LeastSquares(bin_centers[msk], c[msk], np.sqrt(c[msk]), model_counts),
                N=Nevents, **p0)
    mc.limits['f_g'] = (0, 1); mc.limits['f_f'] = (0, 1)
    mc.limits['sigma'] = (0.1, None); mc.limits['tau'] = (0.1, None)
    mc.migrad()
    toy_fg_chi2[i] = mc.values['f_g']
    toy_chi2[i]    = mc.fval
    toy_ndof[i]    = int(msk.sum() - mc.nfit)

### The simulated GoF distribution and the p-value

We histogram the $-2\ln L_\mathrm{min}$ from the toys and mark where our real data falls.
The p-value is the fraction of the distribution to the right of the observed value.

The $-2\ln L_\mathrm{min}$ value is a sum over ~1000 events, so by the central limit
theorem its sampling distribution is expected to be close to **Gaussian**. We therefore
also **fit a Gaussian** to the toy distribution: if it describes the histogram well, the
p-value can equally be read off as $1-\Phi$ of the standardised observed value — a useful
cross-check, and a handy shortcut when running thousands of toys is expensive.

In [ ]:
p_sim = np.mean(toy_2lnL >= T_obs)

# Fit the distribution of -2lnL values with a Gaussian (unbinned ML fit of the toy values):
mg = Minuit(cost.UnbinnedNLL(toy_2lnL, lambda x, mean, sd: stats.norm.pdf(x, mean, sd)),
            mean=toy_2lnL.mean(), sd=toy_2lnL.std())
mg.limits['sd'] = (1e-3, None)
mg.migrad()
g_mean, g_sd = mg.values['mean'], mg.values['sd']

fig, ax = plt.subplots(figsize=(12, 6))
n, edges, _ = ax.hist(toy_2lnL, bins=50, histtype='stepfilled', alpha=0.6, label=f'{N_toy} toy fits')
xx = np.linspace(edges[0], edges[-1], 400)
ax.plot(xx, N_toy*(edges[1]-edges[0])*stats.norm.pdf(xx, g_mean, g_sd), 'C1-', lw=2,
        label=f'Gaussian fit\nmean={g_mean:.0f}, sd={g_sd:.1f}')
ax.axvline(T_obs, color='red', lw=2, label=f'Observed = {T_obs:.1f}')
ax.set(xlabel=r'$-2\ln L$ at the minimum', ylabel='Number of toys',
       title=f'Likelihood goodness-of-fit by simulation:  p = {p_sim:.3f}')
ax.legend()
fig.tight_layout()

p_gauss = stats.norm.sf(T_obs, g_mean, g_sd)
print(f"Simulated GoF p-value (likelihood, direct count)  = {p_sim:.3f}")
print(f"Same from the fitted Gaussian (1 - Phi at T_obs)  = {p_gauss:.3f}")
print(f"GoF p-value from chi2 fit                         = {chi2_prob:.3f}")

### Reading the result

The data really *were* drawn from an exponential, so the fit is genuinely good — and the
simulated likelihood p-value reflects that: it is a perfectly ordinary value, nowhere near
zero. The observed $-2\ln L$ sits right in the bulk of the toy distribution.

Compare this with the printout above. Two things are worth noticing:

- The likelihood fit gave us *no* p-value on its own — we had to **simulate** to calibrate
  $-2\ln L$ into one, because its value is not $\chi^2$-distributed (its mean is nowhere
  near any $N_\mathrm{dof}$).
- The $\chi^2$ fit's "free" p-value can be considerably **smaller and less reliable** for
  the very same data. Two effects pile up here: the binned $\chi^2$ is a worse estimator in
  the sparse tail (we quantify this in Section 5), *and* its built-in p-value leans on the
  assumption that $\chi^2$ follows a $\chi^2$ distribution — an assumption that frays when
  bins have few counts (we test this directly in Section 6).

The simulation method, by contrast, is completely general: it works for *any* fit
statistic, with no assumptions about distributions or degrees of freedom. The only cost is
CPU time — a thousand fits instead of one.

## 5. Why bother with likelihood? The low-statistics tail

We generated all those toys with the *true* answer ($\mathbf{par} = \hat{\mathbf{par}}
\approx \mathbf{par}_\mathrm{true}$), so we can use them to compare the two estimators
honestly. We look at the **signal fraction $f_g$** — the size of the faint peak that lives
in the sparse tail. A good estimator is **unbiased** (mean near the truth) and **precise**
(small spread); the bottom line is the **total error**, the RMS distance from the truth.

The unbinned likelihood uses each event's exact position, so it stays essentially
**unbiased** even for this faint peak. The binned $\chi^2$ must drop the empty bins and
trusts Gaussian $\sqrt{n}$ errors on bins holding just one or two counts — this **biases
the faint-peak estimate low** and gives it a larger total (RMS) error.

In [ ]:
fg_true = par_true[0]
print(f"True f_g = {fg_true}\n")
for name, arr in [('Unbinned LL', toy_fg_ull), ('Chi2 (binned)', toy_fg_chi2)]:
    bias = arr.mean() - fg_true
    rmse = np.sqrt(np.mean((arr - fg_true)**2))   # total error from the truth
    print(f"{name:14s}:  mean = {arr.mean():.4f}   bias = {bias:+.4f}"
          f"   std = {arr.std(ddof=1):.4f}   rms-error = {rmse:.4f}")

lo = min(toy_fg_ull.min(), toy_fg_chi2.min())
hi = max(toy_fg_ull.max(), toy_fg_chi2.max())
bins = np.linspace(lo, hi, 60)
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(toy_fg_ull,  bins=bins, histtype='step', lw=2, label='Unbinned LL')
ax.hist(toy_fg_chi2, bins=bins, histtype='step', lw=2, label='Chi2 (binned)')
ax.axvline(fg_true, color='k', ls=':', lw=2, label='True f_g')
ax.set(xlabel=r'Fitted signal fraction $f_g$', ylabel='Number of toys',
       title='Estimator comparison across the toys')
ax.legend()
fig.tight_layout()

## 6. Bonus: is the $\chi^2$ fit's p-value even trustworthy here?

The $\chi^2$ p-value relies on the assumption that the $\chi^2$ statistic really
follows a $\chi^2$ distribution with $N_\mathrm{dof}$ degrees of freedom. That assumption
is itself only an approximation, and it **breaks down precisely when bins have few
counts**. We already have the toys, so we can simply *check* it: histogram the toy $\chi^2$
values and overlay the textbook $\chi^2$ distribution.

If the two disagree, then even the "automatic" $\chi^2$ p-value should really be calibrated
by simulation — the same tool we just used for the likelihood.

In [ ]:
mean_ndof = toy_ndof.mean()
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(toy_chi2, bins=50, density=True, histtype='stepfilled', alpha=0.6,
        label=f'Toy chi2 values (mean Ndof = {mean_ndof:.0f})')
xx = np.linspace(0, toy_chi2.max(), 400)
ax.plot(xx, stats.chi2.pdf(xx, mean_ndof), 'r-', lw=2,
        label=f'chi2 distribution, Ndof = {mean_ndof:.0f}')
ax.set(xlabel=r'$\chi^2$ of the fit', ylabel='Probability density',
       title='Does the chi2 statistic really follow the chi2 distribution?')
ax.legend()
fig.tight_layout()

print(f"Mean toy chi2 = {toy_chi2.mean():.2f}   vs   mean Ndof = {mean_ndof:.2f}")
print("If these differ noticeably, the textbook chi2 p-value is biased in this")
print("low-statistics regime -> simulation is the safe, general choice.")

## Questions

1. Run the toy loop with `N_toy = 100` and then `N_toy = 1000`. How does the simulated
   p-value (and the smoothness of the distribution) change? With 1000 toys, does the
   direct-count p-value agree with the one read off the fitted Gaussian?

2. Look at the Gaussian fit to the $-2\ln L$ distribution. Its mean is *not* equal to any
   number of degrees of freedom. Convince yourself why the absolute value of $-2\ln L$
   from an unbinned fit cannot be looked up in a $\chi^2$ table — yet its *shape* across
   toys is still close to Gaussian.

3. Compare the bias and spread of $\hat f_g$ from the two fits (Section 5). By how much is
   the binned $\chi^2$ estimate worse? Now reduce `Nbins` to 20 and increase it to 200 —
   what happens to the $\chi^2$ bias, and why?

4. **The real test of a GoF method is that it catches a bad model.** Fit the data with a
   *wrong* model — drop the Gaussian by fixing `m_ull.values['f_g'] = 0` and
   `m_ull.fixed['f_g'] = True` (do the same in the toy loop) — so you fit background only.
   Re-run. Does the simulated likelihood p-value collapse towards zero, as it should?

5. We used $T = -2\ln L_\mathrm{min}$ as the test statistic. Propose an alternative
   statistic that compares the unbinned data to the fitted PDF (e.g. a Kolmogorov–Smirnov
   distance) and obtain its p-value with the same simulation machinery.